# OESD: Oriented Environmental Swiss Dwellings

# Dependencies

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from google.colab import drive
import csv
import os
import math

import shapely.wkt as wkt
import shapely.geometry as sg
from shapely.geometry import Polygon, MultiPolygon, box, LineString, Point
from shapely.ops import transform , unary_union, linemerge #cascaded_union
from shapely import affinity

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import ListedColormap, Normalize, BoundaryNorm, TwoSlopeNorm
from matplotlib.collections import PatchCollection
from matplotlib.cm import ScalarMappable, RdYlGn

from skimage import measure
import random
from PIL import Image

from ipywidgets import widgets
import time
import itertools

# Context Visualization

In [ ]:
#mount google drive
drive.mount ('/content/gdrive', force_remount=True)
folder = "/content/gdrive/MyDrive/O-ESD"
os.chdir(folder)

In [ ]:
#reading OESD
OESD = pd.read_csv('/content/gdrive/MyDrive/O-ESD/OESD.csv')
#drop the unnamed column of OESD
OESD.drop(columns=['Unnamed: 0'], inplace=True)

In [ ]:
#finding data about specific ids
unit_64500 = OESD.loc[OESD['unit_id'] ==64500]
#geopands
gs = gpd.GeoSeries.from_wkt(unit_64500['geometry'])
gdf_test_floor = gpd.GeoDataFrame(unit_64500, geometry=gs, crs=None)
#plot
ax = gdf_test_floor.plot(column='window_orientation', cmap="Set1", legend=True, figsize=(10,10))
#plot the floor layout on the same ax
gdf_test_floor.plot(ax=ax, alpha=0.2, edgecolor='grey', facecolor='none', figsize=(10,10));

In [ ]:
#finding data about specific ids
unit_64500 = unit_df.loc[unit_df['unit_id'] ==64504]
#geopands
gs = gpd.GeoSeries.from_wkt(unit_64500['geometry'])
gdf_test_floor = gpd.GeoDataFrame(unit_64500, geometry=gs, crs=None)
#plot
ax = gdf_test_floor.plot(column='window_orientation', cmap="Set1", legend=True, figsize=(10,10))
#plot the floor layout on the same ax
gdf_test_floor.plot(ax=ax, alpha=0.2, edgecolor='grey', facecolor='none', figsize=(10,10));

In [ ]:
#finding data about specific ids
unit_64500 = unit_df.loc[unit_df['unit_id'] ==64504]
#geopands
gs = gpd.GeoSeries.from_wkt(unit_64500['geometry'])
gdf_test_floor = gpd.GeoDataFrame(unit_64500, geometry=gs, crs=None)

#a datafram called unit_plot with only area entity type
unit_plot = unit_df.loc[unit_df['entity_type'] == 'area']
unit_plot['view_sky_max']=(unit_plot['view_layer_sky']-unit_plot['view_layer_sky'].min())/(unit_plot['view_layer_sky'].max()-unit_plot['view_layer_sky'].min())
#finding areas with highest sky view factor, in this example, the first four
max_ids_sky = unit_plot.iloc[unit_plot['view_sky_max'].argsort()[-3:]]
max_ids_sky = max_ids_sky.sort_values(by= 'view_sky_max', ascending=False)
# Create a new GeoSeries from the geometry of max_ids_sky
gs_sky_area = gpd.GeoSeries.from_wkt(max_ids_sky['geometry'])
gdf_sky_area = gpd.GeoDataFrame(max_ids_sky, geometry=gs_sky_area, crs=None)

#indicator circle placement
# gdf_sky_area['geometry'].centroid
bounds = gdf_test_floor.total_bounds
x_distance = bounds[2] - bounds[0]
y_distance = bounds[3] - bounds[1]
radius_base = 0.08*max(x_distance, y_distance)
# Use .values to get the underlying array of radius values
radius = radius_base*(unit_plot['view_sky_max'].nlargest(3).values)
circle = gdf_sky_area.centroid.buffer(radius, resolution=16)

gc = gpd.GeoSeries(circle)
gdf_circles = gpd.GeoDataFrame(circle, geometry=gc, crs=None)

#overlaying the circle on top of the floor layout
res_symdiff = gdf_circles.overlay(gdf_test_floor, how='identity')
ax = res_symdiff.plot(alpha= 0.5, color = 'Grey', figsize=(10,10))
#gdf_test_floor.plot(ax=ax, facecolor='none');

#plot
identity_6 = gdf_test_floor.overlay(gdf_test_floor, how='identity')
ax = identity_6.plot(ax=ax, alpha= 0.1, edgecolor = 'grey', facecolor= 'none', figsize=(10,10))
#ax = gdf_test_floor.plot(column='window_orientation', cmap="Set1", legend=True)
#plot the floor layout on the same ax
gdf_test_floor.plot(ax=ax, column='window_orientation', cmap="Set1", legend=True, alpha=1, figsize=(10,10));

In [ ]:
#finding data about specific ids
unit_64500 = unit_df.loc[unit_df['unit_id'] ==64504]
#geopands
gs = gpd.GeoSeries.from_wkt(unit_64500['geometry'])
gdf_test_floor = gpd.GeoDataFrame(unit_64500, geometry=gs, crs=None)

#a datafram called unit_plot with only area entity type
unit_plot = unit_df.loc[unit_df['entity_type'] == 'area']
#excluding balcony entity subtype
unit_plot = unit_plot.loc[unit_plot['entity_subtype'] != 'BALCONY']

unit_plot['view_sky_max']=(unit_plot['view_layer_sky']-unit_plot['view_layer_sky'].min())/(unit_plot['view_layer_sky'].max()-unit_plot['view_layer_sky'].min())
unit_plot['view_urban_max']=(unit_plot['view_layer_urban']-unit_plot['view_layer_urban'].min())/(unit_plot['view_layer_urban'].max()-unit_plot['view_layer_urban'].min())
unit_plot['view_landscape_max']=(unit_plot['view_layer_landscape']-unit_plot['view_layer_landscape'].min())/(unit_plot['view_layer_landscape'].max()-unit_plot['view_layer_landscape'].min())
unit_plot['view_ground_max']=(unit_plot['view_layer_ground']-unit_plot['view_layer_ground'].min())/(unit_plot['view_layer_ground'].max()-unit_plot['view_layer_ground'].min())
#same for the column noise_night
unit_plot['noise_night_max']=(unit_plot['noise_night']-unit_plot['noise_night'].min())/(unit_plot['noise_night'].max()-unit_plot['noise_night'].min())
#AIF
unit_plot['AIF_max']=(unit_plot['AIF']-unit_plot['AIF'].min())/(unit_plot['AIF'].max()-unit_plot['AIF'].min())
#MIF
unit_plot['MIF_max']=(unit_plot['MIF']-unit_plot['MIF'].min())/(unit_plot['MIF'].max()-unit_plot['MIF'].min())

bounds = gdf_test_floor.total_bounds
x_distance = bounds[2] - bounds[0]
y_distance = bounds[3] - bounds[1]
radius_base = 0.08*max(x_distance, y_distance)

#number of indicator circles
ncircles_sky = 2
ncircles_urban = 2
ncircles_landscape = 2
ncircles_ground = 2
ncircles_noise = 2
ncircles_MIF = 2
ncircles_AIF = 2

#View Sky
#finding areas with highest sky view factor, in this example, the first four
max_ids_sky = unit_plot.iloc[unit_plot['view_sky_max'].argsort()[-ncircles_sky:]]
max_ids_sky = max_ids_sky.sort_values(by= 'view_sky_max', ascending=False)
# Create a new GeoSeries from the geometry of max_ids_sky
gs_sky_area = gpd.GeoSeries.from_wkt(max_ids_sky['geometry'])
gdf_sky_area = gpd.GeoDataFrame(max_ids_sky, geometry=gs_sky_area, crs=None)


# Use .values to get the underlying array of radius values
radius = radius_base*(unit_plot['view_sky_max'].nlargest(ncircles_sky).values)
circle = gdf_sky_area.centroid.buffer(radius, resolution=16)

gc = gpd.GeoSeries(circle)
gdf_circles_sky = gpd.GeoDataFrame(circle, geometry=gc, crs=None)

identity_1 = gdf_circles_sky.overlay(gdf_test_floor, how='identity')
ax = identity_1.plot(alpha= 0.5, color = 'blue',figsize=(10,10))

#View Landscape
max_ids_landscape = unit_plot.iloc[unit_plot['view_landscape_max'].argsort()[-ncircles_landscape:]]
max_ids_landscape = max_ids_landscape.sort_values(by= 'view_landscape_max', ascending=False)
# Create a new GeoSeries from the geometry of max_ids_sky
gs_landscape_area = gpd.GeoSeries.from_wkt(max_ids_landscape['geometry'])
gdf_landscape_area = gpd.GeoDataFrame(max_ids_landscape, geometry=gs_landscape_area, crs=None)


# Use .values to get the underlying array of radius values
radius = radius_base*(unit_plot['view_landscape_max'].nlargest(ncircles_landscape).values)
circle = gdf_landscape_area.centroid.buffer(radius, resolution=16)

gc = gpd.GeoSeries(circle)
gdf_circles_landscape = gpd.GeoDataFrame(circle, geometry=gc, crs=None)

identity_2 = gdf_circles_landscape.overlay(gdf_circles_sky, how='identity')
ax = identity_2.plot(ax=ax, alpha= 0.5, color = 'green',figsize=(10,10))

#plot
identity_6 = gdf_test_floor.overlay(gdf_test_floor, how='identity')
ax = identity_6.plot(ax=ax, alpha= 0.1, edgecolor = 'grey', facecolor= 'none', figsize=(10,10))
#ax = gdf_test_floor.plot(column='window_orientation', cmap="Set1", legend=True)
#plot the floor layout on the same ax
gdf_test_floor.plot(ax=ax, column='window_orientation', cmap="Set1", legend=True, alpha=1, figsize=(10,10));

In [ ]:
#finding data about specific ids
unit_64500 = unit_df.loc[unit_df['unit_id'] ==64504]
#geopands
gs = gpd.GeoSeries.from_wkt(unit_64500['geometry'])
gdf_test_floor = gpd.GeoDataFrame(unit_64500, geometry=gs, crs=None)

#a datafram called unit_plot with only area entity type
unit_plot = unit_df.loc[unit_df['entity_type'] == 'area']
#excluding balcony entity subtype
unit_plot = unit_plot.loc[unit_plot['entity_subtype'] != 'BALCONY']

unit_plot['view_sky_max']=(unit_plot['view_layer_sky']-unit_plot['view_layer_sky'].min())/(unit_plot['view_layer_sky'].max()-unit_plot['view_layer_sky'].min())
unit_plot['view_urban_max']=(unit_plot['view_layer_urban']-unit_plot['view_layer_urban'].min())/(unit_plot['view_layer_urban'].max()-unit_plot['view_layer_urban'].min())
unit_plot['view_landscape_max']=(unit_plot['view_layer_landscape']-unit_plot['view_layer_landscape'].min())/(unit_plot['view_layer_landscape'].max()-unit_plot['view_layer_landscape'].min())
unit_plot['view_ground_max']=(unit_plot['view_layer_ground']-unit_plot['view_layer_ground'].min())/(unit_plot['view_layer_ground'].max()-unit_plot['view_layer_ground'].min())
#same for the column noise_night
unit_plot['noise_night_max']=(unit_plot['noise_night']-unit_plot['noise_night'].min())/(unit_plot['noise_night'].max()-unit_plot['noise_night'].min())
#AIF
unit_plot['AIF_max']=(unit_plot['AIF']-unit_plot['AIF'].min())/(unit_plot['AIF'].max()-unit_plot['AIF'].min())
#MIF
unit_plot['MIF_max']=(unit_plot['MIF']-unit_plot['MIF'].min())/(unit_plot['MIF'].max()-unit_plot['MIF'].min())

bounds = gdf_test_floor.total_bounds
x_distance = bounds[2] - bounds[0]
y_distance = bounds[3] - bounds[1]
radius_base = 0.08*max(x_distance, y_distance)

#number of indicator circles
ncircles_sky = 2
ncircles_urban = 2
ncircles_landscape = 2
ncircles_ground = 2
ncircles_noise = 2
ncircles_MIF = 2
ncircles_AIF = 2

#AIF
#finding areas with highest sky view factor, in this example, the first four
max_ids_AIF = unit_plot.iloc[unit_plot['AIF_max'].argsort()[-ncircles_AIF:]]
max_ids_AIF = max_ids_AIF.sort_values(by= 'AIF_max', ascending=False)
# Create a new GeoSeries from the geometry of max_ids_sky
gs_AIF_area = gpd.GeoSeries.from_wkt(max_ids_AIF['geometry'])
gdf_AIF_area = gpd.GeoDataFrame(max_ids_AIF, geometry=gs_AIF_area, crs=None)


# Use .values to get the underlying array of radius values
radius = radius_base*(unit_plot['AIF_max'].nlargest(ncircles_AIF).values)
circle = gdf_AIF_area.centroid.buffer(radius, resolution=16)

gc = gpd.GeoSeries(circle)
gdf_circles_AIF = gpd.GeoDataFrame(circle, geometry=gc, crs=None)

identity_1 = gdf_circles_AIF.overlay(gdf_test_floor, how='identity')
ax = identity_1.plot(alpha= 0.5, color = 'yellow',figsize=(10,10))

#Noise
max_ids_noise = unit_plot.iloc[unit_plot['noise_night_max'].argsort()[-ncircles_noise:]]
max_ids_noise = max_ids_noise.sort_values(by= 'noise_night_max', ascending=False)
# Create a new GeoSeries from the geometry of max_ids_sky
gs_noise_area = gpd.GeoSeries.from_wkt(max_ids_noise['geometry'])
gdf_noise_area = gpd.GeoDataFrame(max_ids_noise, geometry=gs_noise_area, crs=None)


# Use .values to get the underlying array of radius values
radius = radius_base*(unit_plot['noise_night_max'].nlargest(ncircles_noise).values)
circle = gdf_noise_area.centroid.buffer(radius, resolution=16)

gc = gpd.GeoSeries(circle)
gdf_circles_noise = gpd.GeoDataFrame(circle, geometry=gc, crs=None)

identity_2 = gdf_circles_noise.overlay(gdf_circles_noise, how='identity')
ax = identity_2.plot(ax=ax, alpha= 0.5, color = 'red',figsize=(10,10))

#plot
identity_6 = gdf_test_floor.overlay(gdf_test_floor, how='identity')
ax = identity_6.plot(ax=ax, alpha= 0.1, edgecolor = 'grey', facecolor= 'none', figsize=(10,10))
#ax = gdf_test_floor.plot(column='window_orientation', cmap="Set1", legend=True)
#plot the floor layout on the same ax
gdf_test_floor.plot(ax=ax, column='window_orientation', cmap="Set1", legend=True, alpha=1, figsize=(10,10));

In [ ]:
#finding data about specific ids
unit_64500 = OESD.loc[OESD['unit_id'] ==64504]
#geopands
gs = gpd.GeoSeries.from_wkt(unit_64500['geometry'])
gdf_test_floor = gpd.GeoDataFrame(unit_64500, geometry=gs, crs=None)

#a datafram called unit_plot with only area entity type
unit_plot = OESD.loc[OESD['entity_type'] == 'area']
#excluding balcony entity subtype
unit_plot = unit_plot.loc[unit_plot['entity_subtype'] != 'BALCONY']

unit_plot['view_sky_max']=(unit_plot['view_layer_sky']-unit_plot['view_layer_sky'].min())/(unit_plot['view_layer_sky'].max()-unit_plot['view_layer_sky'].min())
unit_plot['view_urban_max']=(unit_plot['view_layer_urban']-unit_plot['view_layer_urban'].min())/(unit_plot['view_layer_urban'].max()-unit_plot['view_layer_urban'].min())
unit_plot['view_landscape_max']=(unit_plot['view_layer_landscape']-unit_plot['view_layer_landscape'].min())/(unit_plot['view_layer_landscape'].max()-unit_plot['view_layer_landscape'].min())
unit_plot['view_ground_max']=(unit_plot['view_layer_ground']-unit_plot['view_layer_ground'].min())/(unit_plot['view_layer_ground'].max()-unit_plot['view_layer_ground'].min())
#same for the column noise_night
unit_plot['noise_night_max']=(unit_plot['noise_night']-unit_plot['noise_night'].min())/(unit_plot['noise_night'].max()-unit_plot['noise_night'].min())
#AIF
unit_plot['AIF_max']=(unit_plot['AIF']-unit_plot['AIF'].min())/(unit_plot['AIF'].max()-unit_plot['AIF'].min())
#MIF
unit_plot['MIF_max']=(unit_plot['MIF']-unit_plot['MIF'].min())/(unit_plot['MIF'].max()-unit_plot['MIF'].min())

bounds = gdf_test_floor.total_bounds
x_distance = bounds[2] - bounds[0]
y_distance = bounds[3] - bounds[1]
radius_base = 0.08*max(x_distance, y_distance)

#number of indicator circles
ncircles_sky = 2
ncircles_urban = 2
ncircles_landscape = 2
ncircles_ground = 2
ncircles_noise = 2
ncircles_MIF = 2
ncircles_AIF = 2

#AIF
#finding areas with highest sky view factor, in this example, the first four
max_ids_AIF = unit_plot.iloc[unit_plot['AIF_max'].argsort()[-ncircles_AIF:]]
max_ids_AIF = max_ids_AIF.sort_values(by= 'AIF_max', ascending=False)
# Create a new GeoSeries from the geometry of max_ids_sky
gs_AIF_area = gpd.GeoSeries.from_wkt(max_ids_AIF['geometry'])
gdf_AIF_area = gpd.GeoDataFrame(max_ids_AIF, geometry=gs_AIF_area, crs=None)


# Use .values to get the underlying array of radius values
radius = radius_base*(unit_plot['AIF_max'].nlargest(ncircles_AIF).values)
circle = gdf_AIF_area.centroid.buffer(radius, resolution=16)

gc = gpd.GeoSeries(circle)
gdf_circles_AIF = gpd.GeoDataFrame(circle, geometry=gc, crs=None)

identity_1 = gdf_circles_AIF.overlay(gdf_test_floor, how='identity')
ax = identity_1.plot(alpha= 0.5, color = 'yellow',figsize=(10,10))

#Noise
max_ids_noise = unit_plot.iloc[unit_plot['noise_night_max'].argsort()[-ncircles_noise:]]
max_ids_noise = max_ids_noise.sort_values(by= 'noise_night_max', ascending=False)
# Create a new GeoSeries from the geometry of max_ids_sky
gs_noise_area = gpd.GeoSeries.from_wkt(max_ids_noise['geometry'])
gdf_noise_area = gpd.GeoDataFrame(max_ids_noise, geometry=gs_noise_area, crs=None)


# Use .values to get the underlying array of radius values
radius = radius_base*(unit_plot['noise_night_max'].nlargest(ncircles_noise).values)
circle = gdf_noise_area.centroid.buffer(radius, resolution=16)

gc = gpd.GeoSeries(circle)
gdf_circles_noise = gpd.GeoDataFrame(circle, geometry=gc, crs=None)

identity_2 = gdf_circles_noise.overlay(gdf_circles_noise, how='identity')
ax = identity_2.plot(ax=ax, alpha= 0.5, color = 'red',figsize=(10,10))

#plot
identity_6 = gdf_test_floor.overlay(gdf_test_floor, how='identity')
ax = identity_6.plot(ax=ax, alpha= 0.5, edgecolor = 'grey', facecolor= 'none', figsize=(10,10))
#ax = gdf_test_floor.plot(column='window_orientation', cmap="Set1", legend=True)
#plot the floor layout on the same ax
gdf_test_floor.plot(ax=ax, column='zoning', cmap="Set1", legend=True, alpha=0.4, figsize=(10,10));


In [ ]:
#finding data about specific ids
unit_64500 = OESD.loc[OESD['unit_id'] ==64504]
#geopands
gs = gpd.GeoSeries.from_wkt(unit_64500['geometry'])
gdf_test_floor = gpd.GeoDataFrame(unit_64500, geometry=gs, crs=None)

#a datafram called unit_plot with only area entity type
unit_plot = OESD.loc[OESD['entity_type'] == 'area']
#excluding balcony entity subtype
unit_plot = unit_plot.loc[unit_plot['entity_subtype'] != 'BALCONY']

unit_plot['view_sky_max']=(unit_plot['view_layer_sky']-unit_plot['view_layer_sky'].min())/(unit_plot['view_layer_sky'].max()-unit_plot['view_layer_sky'].min())
unit_plot['view_urban_max']=(unit_plot['view_layer_urban']-unit_plot['view_layer_urban'].min())/(unit_plot['view_layer_urban'].max()-unit_plot['view_layer_urban'].min())
unit_plot['view_landscape_max']=(unit_plot['view_layer_landscape']-unit_plot['view_layer_landscape'].min())/(unit_plot['view_layer_landscape'].max()-unit_plot['view_layer_landscape'].min())
unit_plot['view_ground_max']=(unit_plot['view_layer_ground']-unit_plot['view_layer_ground'].min())/(unit_plot['view_layer_ground'].max()-unit_plot['view_layer_ground'].min())
#same for the column noise_night
unit_plot['noise_night_max']=(unit_plot['noise_night']-unit_plot['noise_night'].min())/(unit_plot['noise_night'].max()-unit_plot['noise_night'].min())
#AIF
unit_plot['AIF_max']=(unit_plot['AIF']-unit_plot['AIF'].min())/(unit_plot['AIF'].max()-unit_plot['AIF'].min())
#MIF
unit_plot['MIF_max']=(unit_plot['MIF']-unit_plot['MIF'].min())/(unit_plot['MIF'].max()-unit_plot['MIF'].min())

bounds = gdf_test_floor.total_bounds
x_distance = bounds[2] - bounds[0]
y_distance = bounds[3] - bounds[1]
radius_base = 0.08*max(x_distance, y_distance)

#number of indicator circles
ncircles_sky = 2
ncircles_urban = 2
ncircles_landscape = 2
ncircles_ground = 2
ncircles_noise = 2
ncircles_MIF = 2
ncircles_AIF = 2

#MIF
#finding areas with highest sky view factor, in this example, the first four
max_ids_MIF = unit_plot.iloc[unit_plot['MIF_max'].argsort()[-ncircles_MIF:]]
max_ids_MIF = max_ids_MIF.sort_values(by= 'MIF_max', ascending=False)
# Create a new GeoSeries from the geometry of max_ids_sky
gs_MIF_area = gpd.GeoSeries.from_wkt(max_ids_MIF['geometry'])
gdf_MIF_area = gpd.GeoDataFrame(max_ids_MIF, geometry=gs_MIF_area, crs=None)


# Use .values to get the underlying array of radius values
radius = radius_base*(unit_plot['MIF_max'].nlargest(ncircles_MIF).values)
circle = gdf_MIF_area.centroid.buffer(radius, resolution=16)

gc = gpd.GeoSeries(circle)
gdf_circles_MIF = gpd.GeoDataFrame(circle, geometry=gc, crs=None)

identity_1 = gdf_circles_MIF.overlay(gdf_test_floor, how='identity')
ax = identity_1.plot(alpha= 0.5, color = 'yellow',figsize=(10,10))

#Noise
max_ids_noise = unit_plot.iloc[unit_plot['noise_night_max'].argsort()[-ncircles_noise:]]
max_ids_noise = max_ids_noise.sort_values(by= 'noise_night_max', ascending=False)
# Create a new GeoSeries from the geometry of max_ids_sky
gs_noise_area = gpd.GeoSeries.from_wkt(max_ids_noise['geometry'])
gdf_noise_area = gpd.GeoDataFrame(max_ids_noise, geometry=gs_noise_area, crs=None)


# Use .values to get the underlying array of radius values
radius = radius_base*(unit_plot['noise_night_max'].nlargest(ncircles_noise).values)
circle = gdf_noise_area.centroid.buffer(radius, resolution=16)

gc = gpd.GeoSeries(circle)
gdf_circles_noise = gpd.GeoDataFrame(circle, geometry=gc, crs=None)

identity_2 = gdf_circles_noise.overlay(gdf_circles_noise, how='identity')
ax = identity_2.plot(ax=ax, alpha= 0.5, color = 'red',figsize=(10,10))

#plot
identity_6 = gdf_test_floor.overlay(gdf_test_floor, how='identity')
ax = identity_6.plot(ax=ax, alpha= 0.1, edgecolor = 'grey', facecolor= 'none', figsize=(10,10))
gdf_test_floor.plot(ax=ax, column='window_orientation', cmap="Set1", legend=True)
#plot the floor layout on the same ax
#gdf_test_floor.plot(ax=ax, column='zoning', cmap="Set1", legend=True, alpha=0.4, figsize=(10,10));

the above is nice to assess bedroom - AIF and noise
and combining the two you realize the room on right should have been living
idea: guess the zone (based on context vis) - check the zone (gt)

In [ ]:
#finding data about specific ids
unit_64500 = unit_df.loc[unit_df['unit_id'] ==64504]
#geopands
gs = gpd.GeoSeries.from_wkt(unit_64500['geometry'])
gdf_test_floor = gpd.GeoDataFrame(unit_64500, geometry=gs, crs=None)

#a datafram called unit_plot with only area entity type
unit_plot = unit_df.loc[unit_df['entity_type'] == 'area']
#excluding balcony entity subtype
unit_plot = unit_plot.loc[unit_plot['entity_subtype'] != 'BALCONY']

unit_plot['view_sky_max']=(unit_plot['view_layer_sky']-unit_plot['view_layer_sky'].min())/(unit_plot['view_layer_sky'].max()-unit_plot['view_layer_sky'].min())
unit_plot['view_urban_max']=(unit_plot['view_layer_urban']-unit_plot['view_layer_urban'].min())/(unit_plot['view_layer_urban'].max()-unit_plot['view_layer_urban'].min())
unit_plot['view_landscape_max']=(unit_plot['view_layer_landscape']-unit_plot['view_layer_landscape'].min())/(unit_plot['view_layer_landscape'].max()-unit_plot['view_layer_landscape'].min())
unit_plot['view_ground_max']=(unit_plot['view_layer_ground']-unit_plot['view_layer_ground'].min())/(unit_plot['view_layer_ground'].max()-unit_plot['view_layer_ground'].min())

bounds = gdf_test_floor.total_bounds
x_distance = bounds[2] - bounds[0]
y_distance = bounds[3] - bounds[1]
radius_base = 0.08*max(x_distance, y_distance)

#number of indicator circles
ncircles_sky = 2
ncircles_urban = 2
ncircles_landscape = 2
ncircles_ground = 2

#View Sky
#finding areas with highest sky view factor, in this example, the first four
max_ids_sky = unit_plot.iloc[unit_plot['view_sky_max'].argsort()[-ncircles_sky:]]
max_ids_sky = max_ids_sky.sort_values(by= 'view_sky_max', ascending=False)
# Create a new GeoSeries from the geometry of max_ids_sky
gs_sky_area = gpd.GeoSeries.from_wkt(max_ids_sky['geometry'])
gdf_sky_area = gpd.GeoDataFrame(max_ids_sky, geometry=gs_sky_area, crs=None)


# Use .values to get the underlying array of radius values
radius = radius_base*(unit_plot['view_sky_max'].nlargest(ncircles_sky).values)
circle = gdf_sky_area.centroid.buffer(radius, resolution=16)

gc = gpd.GeoSeries(circle)
gdf_circles_sky = gpd.GeoDataFrame(circle, geometry=gc, crs=None)

identity_1 = gdf_circles_sky.overlay(gdf_test_floor, how='identity')
ax = identity_1.plot(alpha= 0.5, color = 'blue',figsize=(10,10))

#View Landscape
max_ids_landscape = unit_plot.iloc[unit_plot['view_landscape_max'].argsort()[-ncircles_landscape:]]
max_ids_landscape = max_ids_landscape.sort_values(by= 'view_landscape_max', ascending=False)
# Create a new GeoSeries from the geometry
gs_landscape_area = gpd.GeoSeries.from_wkt(max_ids_landscape['geometry'])
gdf_landscape_area = gpd.GeoDataFrame(max_ids_landscape, geometry=gs_landscape_area, crs=None)


# Use .values to get the underlying array of radius values
radius = radius_base*(unit_plot['view_landscape_max'].nlargest(ncircles_landscape).values)
circle = gdf_landscape_area.centroid.buffer(radius, resolution=16)

gc = gpd.GeoSeries(circle)
gdf_circles_landscape = gpd.GeoDataFrame(circle, geometry=gc, crs=None)

identity_2 = gdf_circles_landscape.overlay(gdf_circles_sky, how='identity')
ax = identity_2.plot(ax=ax, alpha= 0.5, color = 'green',figsize=(10,10))

#View Ground
max_ids_ground = unit_plot.iloc[unit_plot['view_ground_max'].argsort()[-ncircles_ground:]]
max_ids_ground = max_ids_ground.sort_values(by= 'view_ground_max', ascending=False)
# Create a new GeoSeries from the geometry
gs_ground_area = gpd.GeoSeries.from_wkt(max_ids_ground['geometry'])
gdf_ground_area = gpd.GeoDataFrame(max_ids_ground, geometry=gs_ground_area, crs=None)


# Use .values to get the underlying array of radius values
radius = radius_base*(unit_plot['view_ground_max'].nlargest(ncircles_ground).values)
circle = gdf_ground_area.centroid.buffer(radius, resolution=16)

gc = gpd.GeoSeries(circle)
gdf_circles_ground = gpd.GeoDataFrame(circle, geometry=gc, crs=None)

identity_3 = gdf_circles_ground.overlay(gdf_circles_landscape, how='identity')
ax = identity_3.plot(ax=ax, alpha= 0.5, color = 'yellow',figsize=(10,10))

#View Urban
max_ids_urban = unit_plot.iloc[unit_plot['view_urban_max'].argsort()[-ncircles_urban:]]
max_ids_urban = max_ids_urban.sort_values(by= 'view_urban_max', ascending=False)
# Create a new GeoSeries from the geometry
gs_urban_area = gpd.GeoSeries.from_wkt(max_ids_urban['geometry'])
gdf_urban_area = gpd.GeoDataFrame(max_ids_urban, geometry=gs_urban_area, crs=None)


# Use .values to get the underlying array of radius values
radius = radius_base*(unit_plot['view_urban_max'].nlargest(ncircles_urban).values)
circle = gdf_urban_area.centroid.buffer(radius, resolution=16)

gc = gpd.GeoSeries(circle)
gdf_circles_urban = gpd.GeoDataFrame(circle, geometry=gc, crs=None)

identity_4 = gdf_circles_urban.overlay(gdf_circles_ground, how='identity')
ax = identity_4.plot(ax=ax, alpha= 0.5, color = 'red',figsize=(10,10))


#plot
identity_6 = gdf_test_floor.overlay(gdf_test_floor, how='identity')
ax = identity_6.plot(ax=ax, alpha= 0.1, color= 'white', edgecolor = 'grey', facecolor= 'none', figsize=(10,10))
#ax = gdf_test_floor.plot(column='window_orientation', cmap="Set1", legend=True)
#plot the floor layout on the same ax
gdf_test_floor.plot(ax=ax, column='window_orientation', cmap="Set1", legend=True, alpha=1, figsize=(10,10));

# Batch Plotting

In [ ]:
import pandas as pd
import os
import random

def save_units_with_zoning_and_entrance(
    OESD,
    unit_size,
    num_units,
    floor_number,
    output_folder='O-ESD_training_set/unit_8_csv_files'
):
    required_zonings = {"zone01", "zone02", "zone03", "zone04"}

    # Create output directory
    os.makedirs(output_folder, exist_ok=True)

    # Filter area rows for target floor
    area_floor = OESD[
        (OESD['entity_type'] == 'area') &
        (OESD['floor_number'] == floor_number)
    ]

    # Count area size per unit
    area_counts = (
        area_floor.groupby('unit_id')
        .size()
        .reset_index(name='area_count')
    )

    # Filter by unit size
    valid_units = area_counts[area_counts['area_count'] == unit_size]['unit_id'].tolist()

    selected_units = []

    for unit_id in valid_units:
        unit_df = OESD[OESD['unit_id'] == unit_id]

        # Check zoning diversity (must include all zones)
        all_zones = set(unit_df['zoning'].dropna().unique())
        if not required_zonings.issubset(all_zones):
            continue

        # Check exactly one ENTRANCE_DOOR
        entrance_count = (unit_df['zoning'] == 'ENTRANCE_DOOR').sum()
        if entrance_count != 1:
            continue

        # 🆕 Check exactly one zone01
        #zone01_count = (unit_df['zoning'] == 'zone01').sum()
        #if zone01_count != 1:
            #continue

        selected_units.append(unit_id)

    if not selected_units:
        print("No units found satisfying all required conditions.")
        return

    # Randomly sample desired number of units
    sampled_units = random.sample(selected_units, min(num_units, len(selected_units)))

    print(f"Selected unit_ids: {sampled_units}")

    # Save each unit as a CSV
    for unit_id in sampled_units:
        unit_df = OESD[OESD['unit_id'] == unit_id]

        # Keep only area on target floor
        area_df = unit_df[
            (unit_df['entity_type'] == 'area') &
            (unit_df['floor_number'] == floor_number)
        ]

        # Add separators and openings
        separator_df = unit_df[unit_df['entity_type'] == 'separator']
        opening_df = unit_df[unit_df['entity_type'] == 'opening']

        combined_df = pd.concat([area_df, separator_df, opening_df], ignore_index=True)

        # Save CSV
        csv_path = os.path.join(output_folder, f"unit_{unit_id}_floor{floor_number}.csv")
        combined_df.to_csv(csv_path, index=False)

        print(f"Saved: {csv_path}")


# Example:
save_units_with_zoning_and_entrance(OESD, unit_size=8, num_units=200, floor_number=21)


In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.geometry import Point
import os

def plot_unit_zoning(
    csv_path,
    geometry_col="geometry",
    crs="EPSG:4326",
    save_folder=None
):
    df = pd.read_csv(csv_path)

    # Convert to GeoDataFrame
    if geometry_col in df.columns:
        gdf = gpd.GeoDataFrame(df, geometry=gpd.GeoSeries.from_wkt(df[geometry_col]))
        gdf.set_crs(crs, inplace=True)
    else:
        raise ValueError(f"Geometry column '{geometry_col}' not found.")

    # Setup figure and axis
    fig, ax = plt.subplots(figsize=(10, 10))

    # 🟦 Pastel palette for zoning areas
    pastel_colors = ["#AEC6CF", "#FFB347", "#77DD77", "#FF6961"]
    zoning_categories = ['zone01', 'zone02', 'zone03', 'zone04']
    zoning_colors = {zone: pastel_colors[i] for i, zone in enumerate(zoning_categories)}

    legend_patches = []

    for zone, color in zoning_colors.items():
        subset = gdf[(gdf['entity_type'] == 'area') & (gdf['zoning'] == zone)]
        if not subset.empty:
            subset.plot(ax=ax, color=color, edgecolor='black', linewidth=0.8)
            legend_patches.append(mpatches.Patch(color=color, label=f"{zone}"))

    # ▫ Window orientation color palette
    window_palette = {
        'north': "#6495ED",
        'east': "#FFA07A",
        'south': "#90EE90",
        'west': "#FFD700"
    }

    if 'window_orientation' in gdf.columns:
        for direction, color in window_palette.items():
            subset = gdf[
                (gdf['zoning'] == 'WINDOW') &
                (gdf['window_orientation'].str.lower() == direction)
            ]
            if not subset.empty:
                subset.plot(ax=ax, color=color, edgecolor='black', alpha=1.0)
                legend_patches.append(mpatches.Patch(color=color, label=f"Window ({direction.capitalize()})"))

    # ▫ Other special components
    special_styles = {
        'ENTRANCE_DOOR': ('#000000', 1.0, 'Entrance Door'),
        'DOOR': ('#556B2F', 1.0, 'Door'),
        'WALL': ('#C0C0C0', 0.5, 'Wall')
    }

    for zoning_value, (color, alpha, label) in special_styles.items():
        subset = gdf[gdf['zoning'] == zoning_value]
        if not subset.empty:
            subset.plot(ax=ax, color=color, edgecolor='black', alpha=alpha)
            legend_patches.append(mpatches.Patch(color=color, label=label))

    # 🔄 External legend
    ax.legend(
        handles=legend_patches,
        title="Zoning & Elements",
        loc="upper left",
        bbox_to_anchor=(1.02, 1),
        borderaxespad=0
    )

    # Final plot adjustments
    plot_title = f"Zoning Layout: {os.path.basename(csv_path)}"
    ax.set_title(plot_title)
    ax.set_axis_off()
    plt.tight_layout(rect=[0, 0, 0.8, 1])

    # Save or show
    if save_folder:
        os.makedirs(save_folder, exist_ok=True)
        plot_filename = os.path.splitext(os.path.basename(csv_path))[0] + ".png"
        save_path = os.path.join(save_folder, plot_filename)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"Saved plot to: {save_path}")
    else:
        plt.show()


In [ ]:
def batch_plot_zoning(folder_path, floor_number=None, save_folder="O-ESD_training_set/unit_plots_8"):
    os.makedirs(save_folder, exist_ok=True)

    for file in os.listdir(folder_path):
        if file.endswith('.csv') and (floor_number is None or f"floor{floor_number}" in file):
            csv_path = os.path.join(folder_path, file)
            print(f"Processing: {csv_path}")
            try:
                plot_unit_zoning(csv_path, save_folder=save_folder)
            except Exception as e:
                print(f"Error plotting {file}: {e}")


In [ ]:
batch_plot_zoning('/content/gdrive/MyDrive/swiss_dwellings_v3.0.0/ESD/O-ESD_training_set/unit_8_csv_files')

In [ ]:
#number of files in this directory
import os
folder_path = '/content/gdrive/MyDrive/swiss_dwellings_v3.0.0/ESD/O-ESD_training_set/unit_8_csv_files'
#count the number of files
num_files = len([f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))])
print(f"Number of files in the folder: {num_files}")
#